# CauNagi tutorial

This notebook is a complete, non-executed template for the CauNagi workflow. Update the paths, disease-state labels, concept graph, and iDREM settings for your own experiment before running it.

In [ ]:
from pathlib import Path
import sys

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import torch

# Resolve the repository root whether the notebook is opened from the root or tutorials/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "Main_code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing Main_code/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Main_code import Caunagi

INPUT_DIR = PROJECT_ROOT / "input_data"
WORK_DIR = PROJECT_ROOT / "tutorial_workspace"
TEMP_PATH = WORK_DIR / "temp_path"
MODEL_PATH = WORK_DIR / "model_checkpoints"
RESULTS_PATH = PROJECT_ROOT / "results" / "tutorial_markers"
IDREM_DIR = PROJECT_ROOT / "idrem"

STAGE_COLUMN = "disease_stage"
CELLTYPE_COLUMN = "CellType"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")

## Input data

Each numbered `.h5ad` file represents one disease state, with the filename defining its order. A file may contain multiple cell types, but all cells must belong to that state. The state labels in `disease_idx` must match the values in `adata.obs[STAGE_COLUMN]`.

In [ ]:
# Edit these labels to match the disease states stored in your AnnData files.
DISEASE_STATES = [
    "healthy",
    "early_disease",
    "late_disease",
]
disease_idx = {state: index for index, state in enumerate(DISEASE_STATES)}

stage_files = sorted(INPUT_DIR.glob("*.h5ad"), key=lambda path: int(path.stem))
if len(stage_files) < 2:
    raise ValueError("At least two numbered stage files are required.")

expected_indices = list(range(len(stage_files)))
actual_indices = [int(path.stem) for path in stage_files]
if actual_indices != expected_indices:
    raise ValueError("Stage files must be named 0.h5ad, 1.h5ad, 2.h5ad, and so on.")
if set(disease_idx.values()) != set(expected_indices):
    raise ValueError("disease_idx must contain one consecutive index for every stage file.")

reference_var_names = None
for stage_file in stage_files:
    adata = sc.read_h5ad(stage_file)
    required_columns = {STAGE_COLUMN, CELLTYPE_COLUMN}
    missing_columns = required_columns.difference(adata.obs.columns)
    if missing_columns:
        raise KeyError(f"{stage_file.name} is missing columns: {sorted(missing_columns)}")

    state_values = pd.unique(adata.obs[STAGE_COLUMN].astype(str))
    if len(state_values) != 1:
        raise ValueError(f"{stage_file.name} must contain exactly one disease state.")
    if state_values[0] not in disease_idx:
        raise ValueError(f"Unknown disease state in {stage_file.name}: {state_values[0]}")

    current_var_names = pd.Index(adata.var_names)
    if reference_var_names is None:
        reference_var_names = current_var_names
    elif not current_var_names.equals(reference_var_names):
        raise ValueError(f"Gene names/order differ in {stage_file.name}.")

    print(stage_file.name, adata.shape, state_values[0])

TOTAL_STAGE_NUM = len(stage_files)
print(f"Validated {TOTAL_STAGE_NUM} disease states.")

## Define the causal concepts

`concept_list` must match columns in `adata.obs`. The causal matrix has one extra row and column for CauNagi's unexplained concept. Replace the placeholder matrix with the concept graph used in your study.

In [ ]:
concept_list = [STAGE_COLUMN, CELLTYPE_COLUMN]

# Replace this placeholder with the causal DAG defined for your experiment.
concept_cdag = np.zeros(
    (len(concept_list) + 1, len(concept_list) + 1),
    dtype=np.float32,
)

assert concept_cdag.shape == (len(concept_list) + 1, len(concept_list) + 1)
print("Concepts:", concept_list)
print("Causal graph shape:", concept_cdag.shape)

## Create the CauNagi model

The first iteration requires a new `TEMP_PATH`; CauNagi writes intermediate staged data and iDREM files there. Checkpoint files are stored separately in `MODEL_PATH`.

In [ ]:
if TEMP_PATH.exists():
    raise FileExistsError(
        f"{TEMP_PATH} already exists. Choose a new workspace for iteration 0."
    )

model = Caunagi(
    concept_list=concept_list,
    concept_cdag=concept_cdag,
    total_stage_num=TOTAL_STAGE_NUM,
    device=DEVICE,
    save_and_sample_every=100,
)

## Preprocess the stage data

`process_data()` encodes categorical concepts, optionally log-normalizes expression values, and writes the processed input to `INPUT_DIR`.

In [ ]:
model.process_data(
    data_path=INPUT_DIR,
    temp_path=TEMP_PATH,
    stage_name=STAGE_COLUMN,
    iteration=0,
    celltype_concept_name=CELLTYPE_COLUMN,
    disease_idx=disease_idx,
    log_norm=True,
)

## Configure training, CPO, species, and iDREM

Use `Human` or `Mouse` consistently with the reference files in `IDREM_DIR`. The training step is intentionally guarded below so this notebook does not start a long computation accidentally.

In [ ]:
model.setup_train(
    model_save_path=MODEL_PATH,
    epoch_num=100000,
    training_batch_size=64,
    training_lr=2e-5,
    max_profile_size=2000,
    timesteps=1000,
    seed=888,
)

model.register_species("Human")
model.register_CPO_parameters(
    anchor_neighbors=15,
    max_neighbors=35,
    min_neighbors=10,
    resolution_min=0.8,
    resolution_max=1.5,
)
model.register_iDREM_parameters(
    Minimum_Absolute_Log_Ratio_Expression=0.5,
    Convergence_Likelihood=0.001,
    Minimum_Standard_Deviation=0.5,
)

## Run one CauNagi iteration

Set `RUN_PIPELINE = True` only after checking the paths and iDREM installation. One iteration performs diffusion training, clustering, temporal graph construction, iDREM analysis, gene-weight updates, and staged-dataset construction.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    if not IDREM_DIR.is_dir():
        raise FileNotFoundError(f"iDREM directory not found: {IDREM_DIR}")
    model.run_caunagi(idrem_dir=IDREM_DIR, CPO=True)

## Analyse the completed iteration

The analysis input must be the final iteration's `stagedata` directory. Results are written to a separate results directory.

In [ ]:
if RUN_PIPELINE:
    completed_iteration = model.iteration - 1
    stagedata_dir = TEMP_PATH / str(completed_iteration) / "stagedata"
    RESULTS_PATH.mkdir(parents=True, exist_ok=True)

    analysis_adata = model.analyse_UNAGI(
        data_path=stagedata_dir,
        iteration=completed_iteration,
        progressionmarker_background_sampling_times=1000,
        save_dir=RESULTS_PATH,
    )
    print(analysis_adata)

## Run a later iteration

After a successful iteration, call `process_data()` with the next iteration number. CauNagi then reads the previous iteration's staged dataset and reuses the updated `geneWeight` layer.

In [ ]:
# Example for the next iteration. Run only after iteration 0 has completed.
NEXT_ITERATION = 1

if RUN_PIPELINE and model.iteration == NEXT_ITERATION:
    model.process_data(
        data_path=INPUT_DIR,
        temp_path=TEMP_PATH,
        stage_name=STAGE_COLUMN,
        iteration=NEXT_ITERATION,
        celltype_concept_name=CELLTYPE_COLUMN,
        disease_idx=disease_idx,
        log_norm=True,
    )
    model.setup_train(
        model_save_path=MODEL_PATH,
        epoch_num=100000,
        training_batch_size=64,
        training_lr=2e-5,
        max_profile_size=2000,
        timesteps=1000,
    )
    model.run_caunagi(idrem_dir=IDREM_DIR, CPO=True)